# SGP Kits Statistics Report

This notebook analyzes the normalized kit statistics extracted from `command_storage.dat`.

The main focus is kit-level performance. Player IDs are kept as an explanatory dimension so we can detect cases where a kit's aggregate kill count is heavily driven by one or a few players.

## Setup

Imports, kit-name mapping, data loading, and validation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


DATA_DIR = Path("data")

KIT_NAMES = [
    "pigeon",
    "combattant",
    "archer",
    "vindicateur",
    "pyromane",
    "tank",
    "roi",
    "eclaireur",
    "alchimiste",
    "enderman",
    "cancer",
    "poseidon",
]

KIT_ID_TO_NAME = dict(enumerate(KIT_NAMES))
KIT_ORDER = KIT_NAMES


# Load extracted data
kills = pd.read_parquet(DATA_DIR / "kills.parquet")
abilities = pd.read_parquet(DATA_DIR / "abilities.parquet")


# Validate schemas
def require_columns(df: pd.DataFrame, required: set[str], name: str) -> None:
    missing = required - set(df.columns)
    assert not missing, f"{name} is missing columns: {sorted(missing)}"


require_columns(
    kills,
    {"id_killer", "kit_id_killer", "kit_id_victim", "kills"},
    "kills",
)
require_columns(
    abilities,
    {"id", "kit_id", "ability_use"},
    "abilities",
)


# Normalize types
kills["id_killer"] = kills["id_killer"].astype(str)
kills["kit_id_killer"] = pd.to_numeric(kills["kit_id_killer"], errors="raise").astype(int)
kills["kit_id_victim"] = pd.to_numeric(kills["kit_id_victim"], errors="raise").astype(int)
kills["kills"] = pd.to_numeric(kills["kills"], errors="raise").astype(int)

abilities["id"] = abilities["id"].astype(str)
abilities["kit_id"] = pd.to_numeric(abilities["kit_id"], errors="raise").astype(int)
abilities["ability_use"] = pd.to_numeric(abilities["ability_use"], errors="raise").astype(int)


# Validate values and uniqueness
assert not kills[["id_killer", "kit_id_killer", "kit_id_victim", "kills"]].isna().any().any()
assert not abilities[["id", "kit_id", "ability_use"]].isna().any().any()

assert (kills["kills"] >= 0).all(), "Negative kill count found."
assert (abilities["ability_use"] >= 0).all(), "Negative ability-use count found."

assert kills["kit_id_killer"].isin(KIT_ID_TO_NAME).all(), "Unknown killer kit ID found."
assert kills["kit_id_victim"].isin(KIT_ID_TO_NAME).all(), "Unknown victim kit ID found."
assert abilities["kit_id"].isin(KIT_ID_TO_NAME).all(), "Unknown ability kit ID found."

assert not kills.duplicated(
    ["id_killer", "kit_id_killer", "kit_id_victim"]
).any(), "Duplicate kill-stat rows found."

assert not abilities.duplicated(
    ["id", "kit_id"]
).any(), "Duplicate ability-use rows found."

## Total kills by kit

Switch between total kills per kit and the same totals stacked by player contribution. Player IDs appear only on hover in the stacked view.

In [ ]:
# Prepare kill aggregates used by the kill-focused plots.
player_kit_kills = (
    kills.groupby(["id_killer", "kit_id_killer"], as_index=False)["kills"]
    .sum()
    .rename(columns={"kit_id_killer": "kit_id"})
)

player_kit_kills["kit_name"] = player_kit_kills["kit_id"].map(KIT_ID_TO_NAME)

all_kits = pd.DataFrame({
    "kit_id": list(KIT_ID_TO_NAME.keys()),
    "kit_name": list(KIT_ID_TO_NAME.values()),
})

total_kills_by_kit = (
    player_kit_kills.groupby(["kit_id", "kit_name"], as_index=False)["kills"]
    .sum()
)

total_kills_by_kit = (
    all_kits
    .merge(total_kills_by_kit, on=["kit_id", "kit_name"], how="left")
    .fillna({"kills": 0})
)

total_kills_by_kit["kills"] = total_kills_by_kit["kills"].astype(int)


def concentration_from_counts(
    df: pd.DataFrame,
    group_col: str,
    value_col: str,
) -> pd.DataFrame:
    rows = []

    for group_value, group in df.groupby(group_col):
        values = group.loc[group[value_col] > 0, value_col].sort_values(ascending=False)
        total = values.sum()

        rows.append({
            group_col: group_value,
            "players": int(len(values)),
            "top_player_share": (values.iloc[0] / total) if total else np.nan,
            "top_3_share": (values.iloc[:3].sum() / total) if total else np.nan,
        })

    return pd.DataFrame(rows)


kill_concentration = concentration_from_counts(
    player_kit_kills,
    "kit_id",
    "kills",
)

kit_kill_stats = (
    total_kills_by_kit
    .merge(kill_concentration, on="kit_id", how="left")
    .sort_values("kit_id")
    .reset_index(drop=True)
)

kit_kill_stats["players"] = kit_kill_stats["players"].fillna(0).astype(int)


fig = go.Figure()

# Simple total-kills view.
simple_trace_indices = []

for row in total_kills_by_kit.itertuples(index=False):
    simple_trace_indices.append(len(fig.data))
    fig.add_trace(
        go.Bar(
            x=[row.kit_name],
            y=[row.kills],
            name=row.kit_name,
            showlegend=False,
            visible=True,
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Total kills: %{y}<extra></extra>"
            ),
        )
    )

# Stacked player-contribution view.
stacked_trace_indices = []

for player_id, player_data in player_kit_kills.groupby("id_killer"):
    player_data = (
        all_kits[["kit_id", "kit_name"]]
        .merge(player_data[["kit_id", "kills"]], on="kit_id", how="left")
        .fillna({"kills": 0})
    )

    stacked_trace_indices.append(len(fig.data))
    fig.add_trace(
        go.Bar(
            x=player_data["kit_name"],
            y=player_data["kills"],
            name=str(player_id),
            customdata=np.full((len(player_data), 1), str(player_id)),
            showlegend=False,
            visible=False,
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Player ID: %{customdata[0]}<br>"
                "Kills: %{y}<extra></extra>"
            ),
        )
    )

n_traces = len(fig.data)
simple_visible = [False] * n_traces
stacked_visible = [False] * n_traces

for i in simple_trace_indices:
    simple_visible[i] = True

for i in stacked_trace_indices:
    stacked_visible[i] = True

fig.update_layout(
    title="Total kills by kit",
    xaxis_title="Kit",
    yaxis_title="Kills",
    barmode="group",
    hovermode="closest",
    margin=dict(l=60, r=30, t=115, b=60),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "right",
            "x": 0.5,
            "xanchor": "center",
            "y": 1.20,
            "yanchor": "top",
            "showactive": True,
            "buttons": [
                {
                    "label": "Total kills",
                    "method": "update",
                    "args": [
                        {"visible": simple_visible},
                        {"barmode": "group", "title": "Total kills by kit"},
                    ],
                },
                {
                    "label": "By player",
                    "method": "update",
                    "args": [
                        {"visible": stacked_visible},
                        {
                            "barmode": "stack",
                            "title": "Total kills by kit — player contribution",
                        },
                    ],
                },
            ],
        }
    ],
)

fig.update_xaxes(categoryorder="array", categoryarray=KIT_ORDER)
fig.show()

## Player concentration of kills

The top-player and top-3 shares show whether a kit's aggregate kill count is broadly distributed or dominated by a small number of players.

In [ ]:
concentration_plot = kit_kill_stats.loc[
    kit_kill_stats["kills"] > 0,
    ["kit_name", "top_player_share", "top_3_share"],
].copy()

concentration_plot = concentration_plot.sort_values(
    "top_player_share",
    ascending=False,
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=concentration_plot["kit_name"],
        y=concentration_plot["top_player_share"],
        name="Top player",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Top player share: %{y:.1%}<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Bar(
        x=concentration_plot["kit_name"],
        y=concentration_plot["top_3_share"],
        name="Top 3 players",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Top 3 share: %{y:.1%}<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Kill concentration by kit",
    xaxis_title="Kit",
    yaxis_title="Share of kit kills",
    barmode="group",
)

fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

## Total kills vs. player concentration

Each point is a kit. This separates high-kill kits with broad contribution from high-kill kits whose result is driven mostly by one player.

In [ ]:
scatter_data = kit_kill_stats.loc[
    kit_kill_stats["kills"] > 0,
    [
        "kit_name",
        "kills",
        "players",
        "top_player_share",
        "top_3_share",
    ],
].copy()

fig = px.scatter(
    scatter_data,
    x="kills",
    y="top_player_share",
    text="kit_name",
    hover_name="kit_name",
    hover_data={
        "kills": True,
        "players": True,
        "top_player_share": ":.1%",
        "top_3_share": ":.1%",
        "kit_name": False,
    },
    labels={
        "kills": "Total kills",
        "top_player_share": "Top player's share of kit kills",
        "players": "Players with kills",
        "top_3_share": "Top 3 players' share",
    },
    title="Total kills vs. player concentration",
)

fig.update_traces(textposition="top center")
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

# Matchups

These plots describe observed kill counts between kits. They are not win-rate estimates because the dataset does not contain the number of encounters or time played in each matchup.

## Kill matrix

Rows are killer kits and columns are victim kits. This is the direct view of where each kit's kills came from.

In [ ]:
matchup_counts = (
    kills.groupby(["kit_id_killer", "kit_id_victim"], as_index=False)["kills"]
    .sum()
)

matchup_matrix = (
    matchup_counts
    .pivot(index="kit_id_killer", columns="kit_id_victim", values="kills")
    .reindex(index=range(len(KIT_NAMES)), columns=range(len(KIT_NAMES)), fill_value=0)
    .fillna(0)
    .astype(int)
)

fig = go.Figure(
    go.Heatmap(
        z=matchup_matrix.values,
        x=KIT_ORDER,
        y=KIT_ORDER,
        text=matchup_matrix.values,
        texttemplate="%{text}",
        hovertemplate=(
            "<b>%{y} → %{x}</b><br>"
            "Kills: %{z}<extra></extra>"
        ),
        colorbar=dict(title="Kills"),
    )
)

fig.update_layout(
    title="Kills by killer kit and victim kit",
    xaxis_title="Victim kit",
    yaxis_title="Killer kit",
)
fig.show()

## Directional kill share

For each pair of kits, this shows the share of observed kills going in one direction. A value above 50% means the row kit killed the column kit more often than the reverse.

This is useful for spotting asymmetric observed matchups, but it is still **not a win rate**. Hover also shows the total number of kills observed between the two kits, so low-volume extremes are easy to identify.

In [ ]:
directional_share = np.full(
    (len(KIT_NAMES), len(KIT_NAMES)),
    np.nan,
    dtype=float,
)
pair_totals = np.zeros(
    (len(KIT_NAMES), len(KIT_NAMES)),
    dtype=int,
)

for i in range(len(KIT_NAMES)):
    for j in range(len(KIT_NAMES)):
        if i == j:
            continue

        ij = int(matchup_matrix.loc[i, j])
        ji = int(matchup_matrix.loc[j, i])
        total = ij + ji

        pair_totals[i, j] = total

        if total > 0:
            directional_share[i, j] = ij / total

hover_text = np.empty_like(directional_share, dtype=object)

for i in range(len(KIT_NAMES)):
    for j in range(len(KIT_NAMES)):
        if i == j:
            hover_text[i, j] = f"{KIT_NAMES[i]} vs itself"
        elif np.isnan(directional_share[i, j]):
            hover_text[i, j] = (
                f"{KIT_NAMES[i]} → {KIT_NAMES[j]}<br>"
                "No kills observed in either direction"
            )
        else:
            hover_text[i, j] = (
                f"{KIT_NAMES[i]} → {KIT_NAMES[j]}<br>"
                f"Directional share: {directional_share[i, j]:.1%}<br>"
                f"Pair kills observed: {pair_totals[i, j]}"
            )

fig = go.Figure(
    go.Heatmap(
        z=directional_share,
        x=KIT_ORDER,
        y=KIT_ORDER,
        zmin=0,
        zmax=1,
        zmid=0.5,
        customdata=hover_text,
        hovertemplate="%{customdata}<extra></extra>",
        colorbar=dict(
            title="Share",
            tickformat=".0%",
        ),
    )
)

fig.update_layout(
    title="Directional share of observed kills between kit pairs",
    xaxis_title="Other kit",
    yaxis_title="Row kit",
)
fig.show()

# Ability usage

Ability-use counts are analyzed in the same spirit as kills: first by total volume, then by how concentrated that volume is among players.

## Ability uses by kit

Switch between total uses and player-stacked contributions. Player IDs appear only on hover.

In [ ]:
player_kit_abilities = (
    abilities.loc[abilities["ability_use"] > 0]
    .groupby(["id", "kit_id"], as_index=False)["ability_use"]
    .sum()
)

player_kit_abilities["kit_name"] = player_kit_abilities["kit_id"].map(KIT_ID_TO_NAME)

total_abilities_by_kit = (
    player_kit_abilities.groupby(["kit_id", "kit_name"], as_index=False)["ability_use"]
    .sum()
)

total_abilities_by_kit = (
    all_kits
    .merge(total_abilities_by_kit, on=["kit_id", "kit_name"], how="left")
    .fillna({"ability_use": 0})
)

total_abilities_by_kit["ability_use"] = total_abilities_by_kit["ability_use"].astype(int)

ability_concentration = concentration_from_counts(
    player_kit_abilities.rename(columns={"ability_use": "value"}),
    "kit_id",
    "value",
).rename(
    columns={
        "players": "players_using_ability",
        "top_player_share": "top_player_ability_share",
        "top_3_share": "top_3_ability_share",
    }
)

kit_ability_stats = (
    total_abilities_by_kit
    .merge(ability_concentration, on="kit_id", how="left")
    .sort_values("kit_id")
    .reset_index(drop=True)
)

kit_ability_stats["players_using_ability"] = (
    kit_ability_stats["players_using_ability"].fillna(0).astype(int)
)


fig = go.Figure()

simple_trace_indices = []

for row in total_abilities_by_kit.itertuples(index=False):
    simple_trace_indices.append(len(fig.data))
    fig.add_trace(
        go.Bar(
            x=[row.kit_name],
            y=[row.ability_use],
            name=row.kit_name,
            showlegend=False,
            visible=True,
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Ability uses: %{y}<extra></extra>"
            ),
        )
    )

stacked_trace_indices = []

for player_id, player_data in player_kit_abilities.groupby("id"):
    player_data = (
        all_kits[["kit_id", "kit_name"]]
        .merge(player_data[["kit_id", "ability_use"]], on="kit_id", how="left")
        .fillna({"ability_use": 0})
    )

    stacked_trace_indices.append(len(fig.data))
    fig.add_trace(
        go.Bar(
            x=player_data["kit_name"],
            y=player_data["ability_use"],
            name=str(player_id),
            customdata=np.full((len(player_data), 1), str(player_id)),
            showlegend=False,
            visible=False,
            hovertemplate=(
                "<b>%{x}</b><br>"
                "Player ID: %{customdata[0]}<br>"
                "Ability uses: %{y}<extra></extra>"
            ),
        )
    )

n_traces = len(fig.data)
simple_visible = [False] * n_traces
stacked_visible = [False] * n_traces

for i in simple_trace_indices:
    simple_visible[i] = True

for i in stacked_trace_indices:
    stacked_visible[i] = True

fig.update_layout(
    title="Ability uses by kit",
    xaxis_title="Kit",
    yaxis_title="Ability uses",
    barmode="group",
    hovermode="closest",
    margin=dict(l=60, r=30, t=115, b=60),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "right",
            "x": 0.5,
            "xanchor": "center",
            "y": 1.20,
            "yanchor": "top",
            "showactive": True,
            "buttons": [
                {
                    "label": "Total uses",
                    "method": "update",
                    "args": [
                        {"visible": simple_visible},
                        {"barmode": "group", "title": "Ability uses by kit"},
                    ],
                },
                {
                    "label": "By player",
                    "method": "update",
                    "args": [
                        {"visible": stacked_visible},
                        {
                            "barmode": "stack",
                            "title": "Ability uses by kit — player contribution",
                        },
                    ],
                },
            ],
        }
    ],
)

fig.update_xaxes(categoryorder="array", categoryarray=KIT_ORDER)
fig.show()

## Player concentration of ability usage

This shows whether a kit's total ability usage is broadly distributed or mainly produced by one or a few players.

In [ ]:
ability_concentration_plot = kit_ability_stats.loc[
    kit_ability_stats["ability_use"] > 0,
    [
        "kit_name",
        "top_player_ability_share",
        "top_3_ability_share",
    ],
].copy()

ability_concentration_plot = ability_concentration_plot.sort_values(
    "top_player_ability_share",
    ascending=False,
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=ability_concentration_plot["kit_name"],
        y=ability_concentration_plot["top_player_ability_share"],
        name="Top player",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Top player share: %{y:.1%}<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Bar(
        x=ability_concentration_plot["kit_name"],
        y=ability_concentration_plot["top_3_ability_share"],
        name="Top 3 players",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Top 3 share: %{y:.1%}<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Ability-use concentration by kit",
    xaxis_title="Kit",
    yaxis_title="Share of ability uses",
    barmode="group",
)

fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

# Combined analysis

These plots combine the two types of evidence without assuming that ability uses and kills are directly comparable measures of strength.

## Proportion of players who tried each kit

The denominator is every player appearing anywhere in the extracted kill or ability-use data.

Two signals are shown separately:

- players who used the kit's ability at least once;
- players who made at least one kill with the kit.

They are intentionally not merged into a single definition of “tried”, because each signal can miss some genuine kit usage.

In [ ]:
all_player_ids = set(kills["id_killer"].astype(str)) | set(abilities["id"].astype(str))
n_players = len(all_player_ids)

players_with_kill_by_kit = (
    player_kit_kills.loc[player_kit_kills["kills"] > 0]
    .groupby("kit_id")["id_killer"]
    .nunique()
    .reindex(range(len(KIT_NAMES)), fill_value=0)
)

players_with_ability_by_kit = (
    player_kit_abilities.loc[player_kit_abilities["ability_use"] > 0]
    .groupby("kit_id")["id"]
    .nunique()
    .reindex(range(len(KIT_NAMES)), fill_value=0)
)

if n_players == 0:
    kill_player_proportion = players_with_kill_by_kit.astype(float) * np.nan
    ability_player_proportion = players_with_ability_by_kit.astype(float) * np.nan
else:
    kill_player_proportion = players_with_kill_by_kit / n_players
    ability_player_proportion = players_with_ability_by_kit / n_players

reach = pd.DataFrame({
    "kit_name": KIT_ORDER,
    "used_ability": ability_player_proportion.values,
    "made_kill": kill_player_proportion.values,
    "used_ability_count": players_with_ability_by_kit.values,
    "made_kill_count": players_with_kill_by_kit.values,
})

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=reach["kit_name"],
        y=reach["used_ability"],
        name="Used ability ≥ 1 time",
        customdata=np.column_stack([
            reach["used_ability_count"],
            np.full(len(reach), n_players),
        ]),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Used ability: %{y:.1%}<br>"
            "Players: %{customdata[0]:.0f} / %{customdata[1]:.0f}"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Bar(
        x=reach["kit_name"],
        y=reach["made_kill"],
        name="Made ≥ 1 kill",
        customdata=np.column_stack([
            reach["made_kill_count"],
            np.full(len(reach), n_players),
        ]),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Made a kill: %{y:.1%}<br>"
            "Players: %{customdata[0]:.0f} / %{customdata[1]:.0f}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Proportion of observed players who tried each kit",
    xaxis_title="Kit",
    yaxis_title="Proportion of players",
    barmode="group",
)

fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()

## Kills vs. ability uses

Each point is a kit. This is a descriptive comparison of aggregate kill volume and aggregate ability-use volume; it does not imply that one causes the other or that their ratio has the same meaning across kits.

In [ ]:
combined_totals = (
    all_kits
    .merge(
        total_kills_by_kit[["kit_id", "kills"]],
        on="kit_id",
        how="left",
    )
    .merge(
        total_abilities_by_kit[["kit_id", "ability_use"]],
        on="kit_id",
        how="left",
    )
    .fillna({"kills": 0, "ability_use": 0})
)

combined_totals["kills"] = combined_totals["kills"].astype(int)
combined_totals["ability_use"] = combined_totals["ability_use"].astype(int)

fig = px.scatter(
    combined_totals,
    x="ability_use",
    y="kills",
    text="kit_name",
    hover_name="kit_name",
    hover_data={
        "ability_use": True,
        "kills": True,
        "kit_name": False,
    },
    labels={
        "ability_use": "Ability uses",
        "kills": "Kills",
    },
    title="Kills vs. ability uses by kit",
)

fig.update_traces(textposition="top center")
fig.show()

# Summary

The final plot gives a compact cross-metric profile of every kit. Each row is normalized independently, so color intensity means “high relative to the other kits for this metric”, not that different metrics share a common unit.

In [ ]:
summary = (
    all_kits
    .merge(
        kit_kill_stats[
            ["kit_id", "kills", "players", "top_player_share"]
        ].rename(columns={"players": "players_with_kills"}),
        on="kit_id",
        how="left",
    )
    .merge(
        kit_ability_stats[
            [
                "kit_id",
                "ability_use",
                "players_using_ability",
                "top_player_ability_share",
            ]
        ],
        on="kit_id",
        how="left",
    )
)

summary = summary.merge(
    reach[
        ["kit_name", "made_kill", "used_ability"]
    ],
    on="kit_name",
    how="left",
)

metric_specs = [
    ("kills", "Total kills", "count"),
    ("ability_use", "Ability uses", "count"),
    ("made_kill", "Players with ≥1 kill", "percent"),
    ("used_ability", "Players using ability", "percent"),
    ("top_player_share", "Top-player kill share", "percent"),
    ("top_player_ability_share", "Top-player ability-use share", "percent"),
]

z_rows = []
hover_rows = []

for column, label, kind in metric_specs:
    values = pd.to_numeric(summary[column], errors="coerce").to_numpy(dtype=float)

    finite = values[np.isfinite(values)]
    max_value = finite.max() if finite.size else 0.0

    if max_value > 0:
        normalized = values / max_value
    else:
        normalized = np.zeros_like(values)

    z_rows.append(normalized)

    row_hover = []
    for kit_name, value in zip(summary["kit_name"], values):
        if not np.isfinite(value):
            formatted = "No data"
        elif kind == "percent":
            formatted = f"{value:.1%}"
        else:
            formatted = f"{int(value):,}"

        row_hover.append(
            f"{kit_name}<br>{label}: {formatted}"
        )

    hover_rows.append(row_hover)

fig = go.Figure(
    go.Heatmap(
        z=np.array(z_rows),
        x=summary["kit_name"],
        y=[label for _, label, _ in metric_specs],
        customdata=np.array(hover_rows),
        zmin=0,
        zmax=1,
        hovertemplate="%{customdata}<extra></extra>",
        colorbar=dict(
            title="Relative level",
            tickvals=[0, 0.5, 1],
            ticktext=["Low", "Mid", "Highest"],
        ),
    )
)

fig.update_layout(
    title="Kit profile across the main report metrics",
    xaxis_title="Kit",
    yaxis_title="",
)
fig.show()